(sorting)=

# Sorting

```{contents}
:local:
:depth: 2
```

```{index} Compare the Invariant That Each Sort Maintains
```



```{index} Compare the Invariant That Each Sort Maintains
```

## Compare the Invariant That Each Sort Maintains



Selection sort, insertion sort and bubble sort can all order the same small invoice array, but they establish progress differently. Selection chooses the minimum from the unprocessed suffix and fixes one output position. Insertion preserves a sorted processed prefix while placing its next item. Bubble passes adjacent values so that one extreme settles at the remaining boundary. Trace that progress claim rather than memorizing three loops as unrelated syntax.

| Algorithm | Established region after progress | What a trace should show |
| --- | --- | --- |
| Selection | Prefix contains the smallest required values | Minimum search and placement |
| Insertion | Processed prefix is ordered | Saved item, shifts and insertion |
| Bubble | Boundary extreme is settled | Adjacent comparisons and last relevant exchange |

Operation counts depend on which operation is measured. Swapping two elements is not the same event as shifting one value, and comparing loop bounds is not an element-comparer call. A comparison plot that counts swaps for one algorithm and assignments for another does not compare the same unit.

Stability is visible only when equal keys belong to distinguishable records. Strict-shift insertion can preserve tie order, while an ordinary nonadjacent selection swap can move an equal-key record past another. Check labeled IDs as well as sorted amounts. Sorting 2, 2 alone cannot show which occurrence changed position.

Guided prediction: use three labeled records with totals 2-A, 2-B, 1-C. Trace a conventional selection swap into the first position and inspect the final relative order of A and B. Then trace strict-shift insertion. Explain why both arrays can satisfy numeric ordering while only one preserves the original tie order.

For related explanation, see the further reading below.[^development-reading]


Sorting produces an ordered permutation of the input. The algorithms here mutate an array in place. Their counted key comparisons exclude loop-bound checks; swaps and insertion shifts are reported separately. All methods use the same explicit comparer.

```{index} Swap and Preservation
```

## Swap and Preservation




```{index} Swap and Preservation
```

A swap exchanges two positions without losing a value. A temporary variable or tuple assignment preserves both originals; assigning values[a]=values[b] and then values[b]=values[a] would duplicate one value. Swapping a position with itself is harmless but adds unnecessary work.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={10,20,30};
(values[0],values[2])=(values[2],values[0]);
Console.WriteLine(string.Join(", ",values));

30, 20, 10


The elementary sorts use Θ(1) auxiliary state beyond the input. A comparison policy must be consistent; compare integer keys with CompareTo rather than subtracting them, which can overflow. An in-place routine changes the supplied array, so clone it when preserving arrival order matters.

**Common mistake.** Writing the swap as `values[a] = values[b]; values[b] = values[a];`. After the first assignment the original `values[a]` is gone, and the array ends with two copies of `values[b]`; the tuple assignment reads both before writing either.


```{index} Selection Sort and Its Prefix Invariant
```

## Selection Sort and Its Prefix Invariant




```{index} Selection Sort and Its Prefix Invariant
```

At the start of pass s, [0,s) contains the smallest s input occurrences in sorted order. Scan [s,n), retain the smallest index, then swap it into s. The inner-loop invariant tracks the smallest value in the scanned suffix prefix.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={5,2,9,1,3};
var counts=ElementarySort.Selection(values,Comparer<int>.Default);
Console.WriteLine($"sorted={string.Join(",",values)}, comparisons={counts.Comparisons}, swaps={counts.Swaps}");
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


sorted=1,2,3,5,9, comparisons=10, swaps=2


The next chosen minimum extends the sorted prefix and leaves its elements no larger than the remaining suffix. Swaps preserve occurrences. There are exactly (n−1)+(n−2)+…+1=n(n−1)/2 key comparisons and at most n−1 actual swaps. Best and worst time are Θ(n²) for growing n, including sorted input. The shown implementation skips self-swaps. A long counter prevents count overflow on moderate large n.

**Visualize it.** OpenDSA's [Selection Sort](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/SelectionSort.html) slideshow marks the scanned minimum and the single swap that extends the prefix on each pass.

**Common mistake.** Swapping as soon as a smaller value is seen during the scan instead of after the scan ends. The prefix still grows, but the extra swaps break the exact count the invariant promises and move elements that will have to move again.


```{index} Code Tracing: Selection Sort Pass by Pass
```

### Code Tracing: Selection Sort Pass by Pass

Trace selection sort on `[29, 10, 14, 37, 13]`. After each pass write the array, the index of the minimum found, and whether a swap happened. Predict the total number of comparisons and swaps, then run the solution.

In [ ]:
// Solution
using System;

int[] a = { 29, 10, 14, 37, 13 };
int comparisons = 0, swaps = 0;
for (int s = 0; s < a.Length - 1; s++)
{
    int min = s;
    for (int i = s + 1; i < a.Length; i++) { comparisons++; if (a[i] < a[min]) min = i; }
    bool swap = min != s;
    if (swap) { (a[s], a[min]) = (a[min], a[s]); swaps++; }
    Console.WriteLine($"pass {s + 1}: minimum at index {min}{(swap ? ", swapped" : ", already in place")} -> [{string.Join(", ", a)}]");
}
Console.WriteLine($"comparisons={comparisons}, swaps={swaps}");

pass 1: minimum at index 1, swapped -> [10, 29, 14, 37, 13]
pass 2: minimum at index 4, swapped -> [10, 13, 14, 37, 29]
pass 3: minimum at index 2, already in place -> [10, 13, 14, 37, 29]
pass 4: minimum at index 4, swapped -> [10, 13, 14, 29, 37]
comparisons=10, swaps=3


Four passes make 4 + 3 + 2 + 1 = 10 comparisons whatever the data, and three swaps: pass 1 swaps 29 with 10, pass 2 swaps 29 with 13, pass 3 finds 14 already in place and makes no swap, and pass 4 swaps 37 with 29. A pass that finds its minimum already in position costs the same comparisons as any other; only the swap is saved. Selection sort never makes more than n - 1 swaps, which is its one advantage over insertion sort when moving records is expensive.

```{index} Insertion Sort, Shifts, and Nearly Sorted Data
```

## Insertion Sort, Shifts, and Nearly Sorted Data




```{index} Insertion Sort, Shifts, and Nearly Sorted Data
```

Before pass i, [0,i) is sorted. Save the current item, shift strictly greater predecessors right, and insert the saved item in the gap. During shifting the saved value is outside the array; include it when reasoning about occurrence preservation.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={3,1,2};
for(int i=1;i<values.Length;i++)
{
 int current=values[i],j=i-1;
 Console.WriteLine($"pass={i}, saved={current}, before=[{string.Join(",",values)}]");
 while(j>=0 && values[j]>current)
 {
  values[j+1]=values[j];j--;
  Console.WriteLine($"shift; gap={j+1}, array=[{string.Join(",",values)}], saved={current}");
 }
 values[j+1]=current;
 Console.WriteLine($"insert at {j+1}: [{string.Join(",",values)}]");
}

pass=1, saved=1, before=[3,1,2]
shift; gap=0, array=[3,3,2], saved=1
insert at 0: [1,3,2]
pass=2, saved=2, before=[1,3,2]
shift; gap=1, array=[1,3,3], saved=2
insert at 1: [1,2,3]


Equal keys are not shifted, preserving their arrival order. The sorted best case takes n−1 key comparisons and zero shifts; reverse-distinct input takes n(n−1)/2 comparisons and shifts. More precisely, shifts equal the number of inversions: pairs i<j whose earlier key is greater. With I inversions, this insertion implementation takes Θ(n+I) work for n≥1, which explains nearly sorted behavior rather than merely calling it fast.

**Visualize it.** OpenDSA's [Insertion Sort](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/InsertionSort.html) slideshow shows each pass's shifts, and its analysis section animates the best, worst, and average comparison counts claimed here.

**Common mistake.** Shifting with `>=` instead of `>`. Equal keys are then moved past each other on every pass, the sort is no longer stable, and the shift count no longer equals the number of inversions.


```{index} Bubble Sort and a Settled Suffix
```

## Bubble Sort and a Settled Suffix




```{index} Bubble Sort and a Settled Suffix
```

Each left-to-right pass moves the largest remaining item to the current end. The suffix beyond end is already sorted and in final positions; shorten the next pass. If a pass makes no swaps, the remaining prefix has no adjacent inversions and is sorted.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int[] values in new[]{new[]{1,2,3,4},new[]{4,3,2,1},Array.Empty<int>()})
{
 var counts=ElementarySort.Bubble(values,Comparer<int>.Default);
 Console.WriteLine($"[{string.Join(",",values)}]: comparisons={counts.Comparisons}, swaps={counts.Swaps}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


[1,2,3,4]: comparisons=3, swaps=0
[1,2,3,4]: comparisons=6, swaps=6
[]: comparisons=0, swaps=0


Strictly out-of-order swaps make this bubble variant stable. The no-swap exit gives Θ(n) best-case time on sorted inputs; the shrinking-end reverse case has n(n−1)/2 comparisons and swaps, hence Θ(n²). Empty/singleton arrays need no passes. These claims apply to this variant, not to a fixed-pass or full-width bubble implementation.

**Visualize it.** OpenDSA's [Bubble Sort](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/BubbleSort.html) slideshow animates the adjacent swaps that carry the largest remaining item to the settled suffix.

**Common mistake.** Letting every pass run to the full end of the array. The sort still produces the right order, but it compares against the already settled suffix on every pass, and the early-exit saving on sorted input disappears.


```{index} Stability Is Visible on Records
```

## Stability Is Visible on Records




```{index} Stability Is Visible on Records
```

Selection sort’s long-distance swap can reverse equal keys even though the minimum scan uses a strict comparison. Insertion and bubble preserve ties in their shown forms. Observe IDs as well as amounts.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var original=new[]{("A",200),("B",200),("C",100)};
var comparer=Comparer<(string,int)>.Create((a,b)=>a.Item2.CompareTo(b.Item2));
foreach(string algorithm in new[]{"selection","insertion","bubble"})
{
 var copy=original.ToArray();
 if(algorithm=="selection")ElementarySort.Selection(copy,comparer);
 else if(algorithm=="insertion")ElementarySort.Insertion(copy,comparer);
 else ElementarySort.Bubble(copy,comparer);
 Console.WriteLine($"{algorithm}: {string.Join(",",copy.Select(x=>x.Item1))}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


selection: C,B,A
insertion: C,A,B
bubble: C,A,B


Selecting C and swapping it with A gives C,B,A: the two 200-value invoices reverse. Stable output is C,A,B. Stability describes equal comparison keys, not equality of complete records. Do not infer stable behavior from one lucky run of an algorithm without that guarantee.

**Common mistake.** Testing stability with an integer array. Equal integers are indistinguishable, so any output looks stable; the labeled invoices are the only input on which a reversed tie, `C, B, A`, can be seen.


```{index} The Cost of Exchange Sorting
```

### The Cost of Exchange Sorting

Insertion, bubble, and selection sort share one limitation: apart from selection sort's single swap per pass, they move records only between adjacent positions. Each adjacent exchange removes exactly one inversion, so an algorithm that works by adjacent exchanges must perform at least as many exchanges as the input has inversions. A random arrangement of n distinct keys has about n(n - 1) / 4 inversions, half of the maximum, because each pair is as likely to be inverted as not. Any *exchange sort* therefore does Θ(n²) work on average, not only in the worst case. Escaping the quadratic class requires moving records over long distances in one step, which is exactly what Shell sort's gaps, merge sort's merges, and quicksort's partition swaps do.

| | Insertion | Bubble (early exit) | Selection |
| --- | --- | --- | --- |
| Comparisons, best | Θ(n) | Θ(n) | Θ(n²) |
| Comparisons, average and worst | Θ(n²) | Θ(n²) | Θ(n²) |
| Moves, best | 0 | 0 | Θ(n) swaps at most |
| Moves, average and worst | Θ(n²) shifts | Θ(n²) swaps | Θ(n) swaps |
| Stable as shown | Yes | Yes | No |

The example estimates the average inversion count by generating random permutations and compares it with n(n - 1) / 4.

In [ ]:
using System;
using System.Linq;

var random = new Random(2026);
foreach (int n in new[] { 8, 32, 128 })
{
    long total = 0; int trials = 200;
    for (int t = 0; t < trials; t++)
    {
        int[] p = Enumerable.Range(0, n).OrderBy(_ => random.Next()).ToArray();
        for (int i = 0; i < n; i++) for (int j = i + 1; j < n; j++) if (p[i] > p[j]) total++;
    }
    Console.WriteLine($"n={n,3}: average inversions over {trials} random orders = {(double)total / trials,8:F1}   n(n-1)/4 = {n * (n - 1) / 4.0,8:F1}   maximum = {n * (n - 1) / 2}");
}

n=  8: average inversions over 200 random orders =     14.0   n(n-1)/4 =     14.0   maximum = 28
n= 32: average inversions over 200 random orders =    249.0   n(n-1)/4 =    248.0   maximum = 496
n=128: average inversions over 200 random orders =   4054.9   n(n-1)/4 =   4064.0   maximum = 8128


The measured averages sit on the n(n - 1) / 4 line at every size, half the maximum. An insertion sort on typical unsorted data therefore performs about n² / 4 shifts, a quarter of the reversed worst case but still quadratic; doubling the data quadruples the work. The table's best-case column is why insertion sort survives as the finishing step inside faster algorithms: on nearly sorted input the inversion count, not n², is the cost.

Library sorting, mutation, and stable report ordering are taught in {doc}`Chapter 14 <../16_searching_sorting/1601_searching>`. Use those contracts when comparing these optional implementations.


Select by contract and workload: insertion can suit tiny or nearly sorted teaching inputs, but quadratic sorts scale poorly in the worst case. Chapter 22’s stable merge sort has Θ(n log n) time and Θ(n) auxiliary storage. The library algorithms are the usual starting point for application code; custom comparisons still require a consistent order. Mutable records are not deep-cloned by making an array copy.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Selection sort on sorted input still makes n(n - 1) / 2 comparisons. Why can it not stop early the way bubble sort does?

```{dropdown} Answer
Its invariant is about the prefix holding the smallest values, and establishing the next minimum requires scanning the whole remaining suffix regardless of order; nothing in a pass reveals that the suffix is already sorted. Bubble sort's pass does reveal it, by making no swaps.
```

2. Records are large and comparisons cheap. Which elementary sort fits, and what is its cost in each operation?

```{dropdown} Answer
Selection sort: at most n - 1 swaps, so each record moves at most twice, at the price of a fixed n(n - 1) / 2 comparisons. Insertion sort would shift records about n² / 4 times on average.
```

3. Why is an average-case bound for exchange sorts a statement about the problem of exchanging rather than about one algorithm?

```{dropdown} Answer
Because it follows from the inversion count of the input, which no adjacent-exchange algorithm can reduce by more than one per exchange. Any such algorithm, including ones not yet written, does at least n(n - 1) / 4 exchanges on average; the only escape is to move records farther than one position.
```

4. Insertion sort and bubble sort both have Θ(n) best cases. Which inputs achieve it, and are they the same inputs?

```{dropdown} Answer
The same: already sorted input. Insertion makes one failed comparison per item, and bubble's first pass makes no swap and exits. On reversed input both are quadratic; on nearly sorted input insertion is usually cheaper, since a bubble pass still compares every adjacent pair.
```

```{index} Exercise
```

## Exercise




```{index} Exercise
```

Run insertion sort on sorted and reverse arrays of length four. Report comparisons and shifts; relate shifts to inversions.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[][] cases={new[]{1,2,3,4},new[]{4,3,2,1}};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[][] cases={new[]{1,2,3,4},new[]{4,3,2,1}};
foreach(int[] values in cases)
{
 var counts=ElementarySort.Insertion(values,Comparer<int>.Default);
 Console.WriteLine($"comparisons={counts.Comparisons}, shifts={counts.Shifts}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


comparisons=3, shifts=0
comparisons=6, shifts=6


Continue with [Shell Sort and Quicksort](2304_advanced_sorts.ipynb).

```{rubric} Footnotes
```


[^2302_sorting-1]: [Microsoft’s Array.Sort documentation](https://learn.microsoft.com/en-us/dotnet/api/system.array.sort?view=net-10.0) specifies its unstable tie behavior.
[^2302_sorting-2]: [Microsoft’s OrderBy documentation](https://learn.microsoft.com/en-us/dotnet/api/system.linq.enumerable.orderby?view=net-10.0) specifies stable ordering and deferred execution; ToArray materializes the report.

[^development-reading]: Further reading: [OpenDSA, Search in Sorted Arrays](https://opendsa-server.cs.vt.edu/OpenDSA/Books/Catalog/html/SortedSearch.html). The worked business scenarios and guided reasoning here are original teaching examples.
